In [15]:
import pandas as pd
import torch
import os
import pickle

In [6]:
meta_data = pd.read_csv(os.path.join("datasets", "Waterbirds", "metadata.csv"))

meta_data.head(10)

,img_id,img_filename,y,split,place,place_filename
0,1,001.Black_footed_Albatross/Black_Footed_Albatr...,1,2,1,/o/ocean/00002178.jpg
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/l/lake/natural/00000065.jpg
2,3,001.Black_footed_Albatross/Black_Footed_Albatr...,1,2,0,/b/bamboo_forest/00000131.jpg
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/o/ocean/00001268.jpg
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/o/ocean/00003147.jpg
5,6,001.Black_footed_Albatross/Black_Footed_Albatr...,1,2,1,/l/lake/natural/00002698.jpg
6,7,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/o/ocean/00003917.jpg
7,8,001.Black_footed_Albatross/Black_Footed_Albatr...,1,1,1,/o/ocean/00001505.jpg
8,9,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/l/lake/natural/00000560.jpg
9,10,001.Black_footed_Albatross/Black_Footed_Albatr...,1,2,1,/o/ocean/00002073.jpg


In [7]:
# Count how many images are in each split
split_counts = meta_data['split'].value_counts()
print("Number of images in each split:")
print(split_counts)

Number of images in each split:
split
2    5794
0    4795
1    1199
Name: count, dtype: int64


In [14]:
import pandas as pd

meta_data = pd.read_csv(os.path.join("datasets", "Waterbirds", "metadata.csv"))

# Extract folder, e.g. "001.Black_footed_Albatross"
cub_class_name = meta_data["img_filename"].str.split("/").str[0]


# Extract original CUB class number: 1 ... 200
meta_data["cub_class"] = (
    cub_class_name
    .str.split(".")
    .str[0]
    .astype(int) - 1
)

meta_data.head(100)

,img_id,img_filename,y,split,place,place_filename,cub_class
0,1,001.Black_footed_Albatross/Black_Footed_Albatr...,1,2,1,/o/ocean/00002178.jpg,0
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/l/lake/natural/00000065.jpg,0
2,3,001.Black_footed_Albatross/Black_Footed_Albatr...,1,2,0,/b/bamboo_forest/00000131.jpg,0
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/o/ocean/00001268.jpg,0
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,1,0,1,/o/ocean/00003147.jpg,0
...,...,...,...,...,...,...,...
95,96,002.Laysan_Albatross/Laysan_Albatross_0076_671...,1,0,1,/l/lake/natural/00004400.jpg,1
96,97,002.Laysan_Albatross/Laysan_Albatross_0096_673...,1,0,1,/o/ocean/00004001.jpg,1
97,98,002.Laysan_Albatross/Laysan_Albatross_0064_674...,1,2,0,/b/bamboo_forest/00004152.jpg,1
98,99,002.Laysan_Albatross/Laysan_Albatross_0039_924...,1,1,0,/b/bamboo_forest/00001446.jpg,1


### Check that the image IDs /  in waterbirds and CUB is the same

In [ ]:
cub_img_info_file = os.path.join("datasets", "CUB", "CUB_200_2011", "CUB_200_2011", "images.txt")


with open(cub_img_info_file, "r") as f:
    cub_img_info = f.readlines()
    for i in range(len(cub_img_info)):
            
        cub_image_name = cub_img_info[i].strip().split(" ")[1]
        cub_image_id = int(cub_img_info[i].strip().split(" ")[0])
        # Get the corresponding row in the metadata DataFrame by using img_id
        row = meta_data[meta_data['img_id'] == cub_image_id]
        if i == 0:
            print(f"Checking image ID {cub_image_id} with name {cub_image_name}")
        if not row.empty:
            # Check if the image name matches
            if row.iloc[0]['img_filename'] != cub_image_name:
                raise ValueError(f"Image name mismatch for ID {cub_image_id}: {row.iloc[0]['img_filename']} vs {cub_image_name}")
        else: 
            raise ValueError(f"Image ID {cub_image_id} not found in metadata DataFrame.")
        





### Waterbirds script

In [24]:



pkl_dir = os.path.join("datasets", "CUB", "class_attr_data_10")


cub_annotations = {}

for pkl_file in ("train.pkl", "val.pkl", "test.pkl"):
    with open(os.path.join(pkl_dir, pkl_file), "rb") as f:
        for sample in pickle.load(f):
            cub_annotations[sample["id"]] = {
                "class_label": sample["class_label"],
                "attribute_label": sample["attribute_label"],
                "rel_path": "/".join(sample["img_path"].split("/")[-2:])
            }
            
            

for img_id, info in cub_annotations.items():
    # Get the corresponding row in the metadata DataFrame by using img_id
    row = meta_data[meta_data['img_id'] == img_id]
    if not row.empty:
        # Check if the image name matches
        if row.iloc[0]['img_filename'] != info["rel_path"]:
            raise ValueError(f"Image name mismatch for ID {img_id}: {row.iloc[0]['img_filename']} vs {info['rel_path']}")
    else: 
        raise ValueError(f"Image ID {img_id} not found in metadata DataFrame.")

/var/folders/wb/9whmmh7s5bvf174ppr2jwprr0000gn/T/ipykernel_19247/3718808309.py:8: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  for sample in pickle.load(f):
